# Risk classification pipeline (Colab)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Upload `train.xlsx` (left sidebar → Files). Add `GROQ_API_KEY` under Secrets (key icon) and switch on notebook access.

**Run all.** Per-fold results are cached (to Google Drive if you allow it), so after a disconnect just re-run: finished folds are skipped.

The 20% held-out split is scored once, at the end; nothing is tuned on it. All settings are in the CONFIG cell.

In [ ]:
# ============================================================
# 1. INSTALL + IMPORTS + DEVICE
# ============================================================
!pip -q install transformers sentencepiece openpyxl groq

import os, re, ast, gc, html, json, math, time, random, hashlib, threading, warnings
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from scipy.sparse import hstack
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


SEED = 42
set_seed(SEED)

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
DEV_TYPE = DEVICE.type
print("Device:", DEVICE)
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!! No GPU. Runtime > Change runtime type > T4 GPU, then run again.")

In [ ]:
# ============================================================
# 2. CONFIG  -  every knob lives here
# ============================================================
CODE_VERSION = "v1"                  # bump after editing the training logic so old cached folds are ignored
FILE_PATH = "train.xlsx"             # upload via left sidebar > Files (lands in /content)

# ---- what to run ----
RUN_FACTORS = True                   # False = risk only (much faster while you iterate on accuracy)
RUN_EVIDENCE = True                  # False = skip the Groq step
USE_TFIDF_MEMBER = True              # cheap CPU model blended into the risk ensemble
USE_DRIVE_CACHE = True               # per-fold results go to Google Drive -> resumable after a disconnect

# ---- data / split ----
N_FOLDS = 5
MAX_LENGTH = 512                     # tokens (old: 256). Longer posts keep head + tail instead of losing the tail
HEAD_FRAC = 0.5                      # share of the token budget taken from the START of a long post

# ---- risk ensemble: each entry is trained 5-fold, once per seed ----
BACKBONES = {
    "mental/mental-roberta-base": dict(lr=3e-5, batch=8, accum=2),
    "microsoft/deberta-v3-base":  dict(lr=2e-5, batch=4, accum=4),
}
SEEDS = [42]                         # e.g. [42, 7, 2024] = bigger ensemble, 3x the time
FACTOR_BACKBONE = "mental/mental-roberta-base"

# ---- fine-tuning ----
EPOCHS = 5
PATIENCE = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
LLRD = 0.90                          # layer-wise LR decay: lower layers move less than upper layers
HEAD_LR = 2e-4                       # the new classifier layer learns faster than the pretrained encoder
DROPOUT = 0.20
LABEL_SMOOTHING = 0.05
RISK_CLASS_WEIGHT_POWER = 0.5        # 0 = no class weights, 1 = full inverse-frequency (old notebook)
FACTOR_POS_WEIGHT_CAP = 10.0         # old notebook had no cap: a rare factor could get weight 50+
EVAL_BATCH = 32

# ---- evidence ----
GROQ_MODEL = "openai/gpt-oss-120b"
EVIDENCE_WORKERS = 3
EVIDENCE_N = None                    # e.g. 40 = only the first 40 held-out posts (quick test)
EVIDENCE_FALLBACK = True             # LLM gave nothing verbatim -> best-matching sentence (counted separately)
MAX_POST_CHARS = 6000

TARGET_ACC = 0.80

In [ ]:
# ============================================================
# 3. CACHE FOLDER + SECRETS (fail early, not after an hour of training)
# ============================================================
CACHE_DIR = Path("/content/risk_cache")
if USE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CACHE_DIR = Path("/content/drive/MyDrive/risk_pipeline_cache")
    except Exception as e:
        print("Drive not mounted -> caching locally:", str(e)[:100])
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Cache folder:", CACHE_DIR)


def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


groq_client = None
if RUN_EVIDENCE:
    _gkey = get_secret("GROQ_API_KEY")
    if not _gkey:
        print("!! GROQ_API_KEY not found (Colab: left sidebar > Secrets). Evidence step will be skipped.")
        RUN_EVIDENCE = False
    else:
        from groq import Groq
        groq_client = Groq(api_key=_gkey)
        try:
            groq_client.models.list()
            print("Groq key OK")
        except Exception as e:
            if getattr(e, "status_code", None) in (401, 403):
                print("!! Groq rejected the key -> evidence step will be skipped:", str(e)[:120])
                RUN_EVIDENCE = False
            else:
                print("Groq check failed (maybe temporary), keeping the evidence step on:", str(e)[:120])

In [ ]:
# ============================================================
# 4. LOAD DATA, FIND COLUMNS, CLEAN, BUILD LABELS, AUDIT
# ============================================================
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"{FILE_PATH} not found - upload it via the left sidebar (Files); it lands in /content.")

df = pd.read_excel(FILE_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


def find_column(df, possible_names):
    lower_map = {str(c).lower().strip(): c for c in df.columns}
    for name in possible_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    return None


POST_COL     = find_column(df, ["post", "text", "content"])
RISK_COL     = find_column(df, ["suicide risk", "risk", "suicide_risk"])
FACTOR_COL   = find_column(df, ["factors", "factor"])
EVIDENCE_COL = find_column(df, ["evidence for suicide risk level", "evidence", "evidence_for_suicide_risk_level"])
USER_COL     = find_column(df, ["anon_user_id", "user_id", "author", "user"])
print("Detected:", dict(post=POST_COL, risk=RISK_COL, factors=FACTOR_COL, evidence=EVIDENCE_COL, user=USER_COL))
for what, col in (("post", POST_COL), ("risk", RISK_COL), ("factors", FACTOR_COL), ("evidence", EVIDENCE_COL)):
    if col is None:
        raise ValueError(f"Could not find the {what} column in {df.columns.tolist()}")
if USER_COL is None:
    print("!! No user-id column: posts are split one by one, so one user can end up in train AND test.")


# ---------------- text cleaning ----------------
def clean_text(text):
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"</?[a-zA-Z][^>]*>", " ", text)             # real HTML tags only ("<3", "a < b" survive)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # (?<![\w/]) = start of a word. The old pattern r"u/\w+" also hit "you/your" -> "yo[USER]", "or/and" -> "o[SUBREDDIT]"
    text = re.sub(r"(?<![\w/])/?u/[\w-]+", "[USER]", text)
    text = re.sub(r"(?<![\w/])/?r/[\w-]+", "[SUBREDDIT]", text)
    return re.sub(r"\s+", " ", text).strip()


df["text_original"] = df[POST_COL].fillna("").astype(str)
df["text_clean"] = df["text_original"].apply(clean_text)
df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)
print("Rows after cleaning:", len(df))

# ---------------- risk labels ----------------
RISK_ORDER = ["indicator", "ideation", "behavior", "attempt"]      # ordinal: adjacent classes are the usual confusion
RISK_TO_ID = {label: i for i, label in enumerate(RISK_ORDER)}
ID_TO_RISK = {i: label for label, i in RISK_TO_ID.items()}

df["risk_label"] = (df[RISK_COL].astype(str).str.strip().str.lower()
                    .str.replace(r"[_-]", " ", regex=True).str.strip())
unknown = set(df["risk_label"]) - set(RISK_ORDER)
if unknown:
    raise ValueError(f"Unknown risk labels: {unknown}")
df["risk_id"] = df["risk_label"].map(RISK_TO_ID).astype(int)

# ---------------- groups (users) ----------------
# a missing user id becomes its own group instead of one giant "unknown" group that lands in a single fold
if USER_COL is not None:
    groups_all = np.array([str(u) if pd.notna(u) else f"row_{i}" for i, u in enumerate(df[USER_COL])])
else:
    groups_all = np.array([f"row_{i}" for i in range(len(df))])


# ---------------- factors ----------------
def parse_factor_list(x):
    if pd.isna(x):
        return []
    raw = str(x).strip()
    if raw == "":
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    if ";" in raw:
        return [p.strip() for p in raw.split(";") if p.strip()]
    if "," in raw:
        return [p.strip() for p in raw.split(",") if p.strip()]
    return [raw]


df["factor_list"] = df[FACTOR_COL].apply(parse_factor_list)
factor_names = sorted({f for fl in df["factor_list"] for f in fl})
N_FACTORS = len(factor_names)
FACTOR_TO_ID = {f: i for i, f in enumerate(factor_names)}
ID_TO_FACTOR = {i: f for f, i in FACTOR_TO_ID.items()}

Y_factor = np.zeros((len(df), N_FACTORS), dtype=np.float32)
for i, fl in enumerate(df["factor_list"]):
    for f in fl:
        Y_factor[i, FACTOR_TO_ID[f]] = 1
print(f"{N_FACTORS} factors:", factor_names)
if N_FACTORS != 24:
    print(f"!! expected 24 factors but found {N_FACTORS}: check the parsing (commas inside factor names?)")


# ---------------- evidence (gold) ----------------
def parse_evidence_list(x):
    if pd.isna(x):
        return []
    raw = str(x).strip()
    if not raw:
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    return [p.strip() for p in raw.split(";") if p.strip()]


df["evidence_list"] = df[EVIDENCE_COL].apply(parse_evidence_list)

# ---------------- audit ----------------
df["word_count"] = df["text_original"].str.split().str.len()
per_user = pd.Series(groups_all).value_counts()
print(f"\nposts: {len(df)} | users: {len(per_user)} | posts per user: median {per_user.median():.0f}, max {per_user.max()}")
print("class share (%):", (df["risk_label"].value_counts(normalize=True).reindex(RISK_ORDER) * 100).round(1).to_dict())
dup = df.duplicated("text_clean", keep=False)
conflicts = int(df[dup].groupby("text_clean")["risk_id"].nunique().gt(1).sum())
print(f"rows whose text appears more than once: {int(dup.sum())} | of those texts, {conflicts} carry conflicting risk labels")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
counts = df["risk_label"].value_counts().reindex(RISK_ORDER).fillna(0)
ax[0].bar(counts.index, counts.values)
ax[0].set_title("Risk distribution")
ax[1].hist(df["word_count"].clip(upper=df["word_count"].quantile(0.99)), bins=50)
ax[1].set_title("Post length (words, top 1% clipped)")
for r in RISK_ORDER:
    ax[2].hist(df.loc[df["risk_label"] == r, "word_count"].clip(upper=600), bins=40, alpha=0.5, label=r)
ax[2].legend()
ax[2].set_title("Post length by risk level")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5. GROUP-AWARE 80/20 SPLIT, THEN 5-FOLD CV INSIDE THE 80%
# ============================================================
y_all = df["risk_id"].to_numpy()

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
dev_idx, test_idx = next(outer.split(np.zeros(len(df)), y_all, groups_all))      # fold 0 = the 20% held-out

dev_df = df.iloc[dev_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)
y_dev, y_test = y_all[dev_idx], y_all[test_idx]
Y_factor_dev, Y_factor_test = Y_factor[dev_idx], Y_factor[test_idx]
groups_dev, groups_test = groups_all[dev_idx], groups_all[test_idx]
dev_texts, test_texts = dev_df["text_clean"].to_numpy(), test_df["text_clean"].to_numpy()

assert set(groups_dev).isdisjoint(groups_test), "user leakage between development set and held-out test!"
print(f"Development: {len(dev_df)} posts / {len(set(groups_dev))} users | Held-out: {len(test_df)} posts / {len(set(groups_test))} users")
print("Class share dev (%) :", (pd.Series(y_dev).map(ID_TO_RISK).value_counts(normalize=True).reindex(RISK_ORDER) * 100).round(1).to_dict())
print("Class share test (%):", (pd.Series(y_test).map(ID_TO_RISK).value_counts(normalize=True).reindex(RISK_ORDER) * 100).round(1).to_dict())
print("exact same text in dev and held-out:", len(set(dev_texts) & set(test_texts)))

inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(inner.split(np.zeros(len(dev_df)), y_dev, groups_dev))               # computed once, reused by every model
fold_id = np.zeros(len(dev_df), dtype=int)
for k, (tr, va) in enumerate(folds):
    assert set(groups_dev[tr]).isdisjoint(groups_dev[va]), f"user leakage inside CV fold {k}"
    fold_id[va] = k
SPLIT_SIG = hashlib.md5(dev_idx.tobytes() + fold_id.tobytes()).hexdigest()[:8]    # part of every cache file name
print(f"No user appears on both sides of any split. {N_FOLDS} folds, split signature {SPLIT_SIG}")

In [ ]:
# ============================================================
# 6. TOKENISATION, DATASET, MODEL, TRAINER, 5-FOLD RUNNER
# ============================================================
def slug(name):
    return name.split("/")[-1].replace("-", "_")


TOK, ENC = {}, {}                    # tokenizer / encoded ids per backbone (encoded once, reused by every fold)


def load_tokenizer(name):
    try:
        return AutoTokenizer.from_pretrained(name)
    except Exception as e:
        print(f"  fast tokenizer failed for {name} ({str(e)[:80]}); trying the slow one")
        return AutoTokenizer.from_pretrained(name, use_fast=False)


def special_tokens(tok):
    """(prefix, suffix) ids the tokenizer wraps around a text: ([<s>], [</s>]) for RoBERTa, ([CLS], [SEP]) for DeBERTa.
    Probed through tok(...) only, because tokenizer internals (build_inputs_with_special_tokens ...) differ between transformers versions."""
    with_sp = tok("hello world", add_special_tokens=True)["input_ids"]
    without = tok("hello world", add_special_tokens=False)["input_ids"]
    if not without:
        raise ValueError("this tokenizer returns no tokens for a normal sentence - wrong tokenizer for the model?")
    n = len(without)
    for s in range(len(with_sp) - n + 1):
        if with_sp[s:s + n] == without:
            return with_sp[:s], with_sp[s + n:]
    raise ValueError("could not work out how this tokenizer adds special tokens")


def encode_texts(tok, texts):
    """Token ids with head+tail truncation (no padding here: batches are padded to their own longest post)."""
    prefix, suffix = special_tokens(tok)
    budget = MAX_LENGTH - len(prefix) - len(suffix)
    n_head = int(budget * HEAD_FRAC)
    raw = tok(list(texts), add_special_tokens=False, truncation=False,
              return_attention_mask=False, verbose=False)["input_ids"]
    out = []
    for ids in raw:
        if len(ids) > budget:
            ids = ids[:n_head] + ids[len(ids) - (budget - n_head):]
        out.append(prefix + ids + suffix)
    return out, np.array([len(x) for x in raw]), budget


def get_encodings(backbone):
    if backbone not in ENC:
        TOK[backbone] = load_tokenizer(backbone)
        ids_dev, raw_len, budget = encode_texts(TOK[backbone], dev_texts)
        ids_test, _, _ = encode_texts(TOK[backbone], test_texts)
        if np.median(raw_len) < 3:
            raise ValueError(f"{backbone}: posts tokenise to ~0 tokens - check the tokenizer / text column")
        ENC[backbone] = (ids_dev, ids_test)
        print(f"{backbone}: posts are {np.median(raw_len):.0f} tokens (median), {np.percentile(raw_len, 95):.0f} (95th pct); "
              f"{(raw_len > budget).mean():.1%} are longer than {budget} and get cut head+tail")
    return ENC[backbone]


class IdsDataset(Dataset):
    def __init__(self, ids, labels=None):
        self.ids, self.labels = ids, labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], (None if self.labels is None else self.labels[i])


def make_collate(pad_id, label_dtype):
    def collate(batch):
        longest = max(len(x) for x, _ in batch)
        ids = torch.full((len(batch), longest), pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch), longest), dtype=torch.long)
        for i, (x, _) in enumerate(batch):
            ids[i, :len(x)] = torch.tensor(x, dtype=torch.long)
            mask[i, :len(x)] = 1
        out = {"input_ids": ids, "attention_mask": mask}
        if batch[0][1] is not None:
            out["labels"] = torch.tensor(np.stack([b for _, b in batch]), dtype=label_dtype)
        return out
    return collate


class Classifier(nn.Module):
    """Encoder + masked mean pooling + dropout + linear head. Same class for risk (4 outputs) and factors (24)."""

    def __init__(self, backbone, n_out):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone)
        self.dropout = nn.Dropout(DROPOUT)
        self.head = nn.Linear(self.encoder.config.hidden_size, n_out)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state.float()
        m = attention_mask.unsqueeze(-1).float()
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
        return self.head(self.dropout(pooled))


def build_optimizer(model, lr):
    """AdamW with layer-wise LR decay (embeddings smallest LR, top layer = lr), no weight decay on bias/LayerNorm."""
    enc = model.encoder
    n_layers = enc.config.num_hidden_layers
    groups = {}
    for name, p in enc.named_parameters():
        m = re.search(r"encoder\.layer\.(\d+)\.", name)
        if m:
            depth = int(m.group(1))
        elif name.startswith("embeddings."):
            depth = -1
        else:
            depth = n_layers - 1                      # pooler / relative-position tables / final norms: top LR
        no_decay = name.endswith(".bias") or "LayerNorm" in name or "layer_norm" in name
        groups.setdefault((depth, no_decay), []).append(p)
    param_groups = [{"params": ps, "lr": lr * LLRD ** (n_layers - 1 - depth),
                     "weight_decay": 0.0 if no_decay else WEIGHT_DECAY}
                    for (depth, no_decay), ps in groups.items()]
    param_groups.append({"params": [model.head.weight], "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY})
    param_groups.append({"params": [model.head.bias], "lr": HEAD_LR, "weight_decay": 0.0})
    return torch.optim.AdamW(param_groups)


def class_weights(y):
    counts = np.bincount(y, minlength=4)
    w = (len(y) / (4 * np.maximum(counts, 1))) ** RISK_CLASS_WEIGHT_POWER      # computed from the TRAINING part of the fold only
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def pos_weights(Y):
    pos = Y.sum(0)
    return torch.tensor(np.minimum((len(Y) - pos) / np.maximum(pos, 1), FACTOR_POS_WEIGHT_CAP),
                        dtype=torch.float32, device=DEVICE)


@torch.no_grad()
def predict_probs(model, ids_list, task, pad_id):
    model.eval()
    order = np.argsort([len(x) for x in ids_list])                # similar lengths together = little padding
    loader = DataLoader(IdsDataset([ids_list[i] for i in order]), batch_size=EVAL_BATCH,
                        shuffle=False, collate_fn=make_collate(pad_id, torch.long))
    chunks = []
    for batch in loader:
        with torch.autocast(device_type=DEV_TYPE, dtype=torch.float16, enabled=USE_CUDA):
            logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
        logits = logits.float()
        chunks.append((torch.softmax(logits, 1) if task == "risk" else torch.sigmoid(logits)).cpu().numpy())
    sorted_probs = np.vstack(chunks)
    out = np.empty_like(sorted_probs)
    out[order] = sorted_probs                                     # undo the length sort
    return out


def train_model(task, backbone, hp, ids_dev, ids_test, labels, tr, va, seed):
    """Fine-tune on `tr`, keep the best epoch on `va` (risk: accuracy, factors: macro-F1). Returns val probs, test probs."""
    set_seed(seed)
    pad_id = TOK[backbone].pad_token_id
    is_risk = task == "risk"
    n_out = 4 if is_risk else labels.shape[1]
    collate = make_collate(pad_id, torch.long if is_risk else torch.float32)

    model = Classifier(backbone, n_out).to(DEVICE)
    if is_risk:
        criterion = nn.CrossEntropyLoss(weight=class_weights(labels[tr]), label_smoothing=LABEL_SMOOTHING)
    else:
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights(labels[tr]))

    tr_loader = DataLoader(IdsDataset([ids_dev[i] for i in tr], labels[tr]), batch_size=hp["batch"],
                           shuffle=True, collate_fn=collate, pin_memory=USE_CUDA)
    ids_va, y_va = [ids_dev[i] for i in va], labels[va]
    accum = hp["accum"]
    total_steps = math.ceil(len(tr_loader) / accum) * EPOCHS
    optimizer = build_optimizer(model, hp["lr"])
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_CUDA)

    best_key, best_state, best_probs, best_epoch, bad = None, None, None, 0, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tqdm(tr_loader, desc=f"{task} epoch {epoch}/{EPOCHS}", leave=False)):
            ids = batch["input_ids"].to(DEVICE)
            am = batch["attention_mask"].to(DEVICE)
            y = batch["labels"].to(DEVICE)
            with torch.autocast(device_type=DEV_TYPE, dtype=torch.float16, enabled=USE_CUDA):
                loss = criterion(model(ids, am).float(), y) / accum
            scaler.scale(loss).backward()
            # also step on the last, possibly short, accumulation window (old code leaked it into the next epoch)
            if (step + 1) % accum == 0 or step + 1 == len(tr_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        probs = predict_probs(model, ids_va, task, pad_id)
        if is_risk:
            key = (accuracy_score(y_va, probs.argmax(1)), -log_loss(y_va, probs, labels=[0, 1, 2, 3]))   # accuracy, then lower loss
        else:
            key = (f1_score(y_va.astype(int), (probs >= 0.5).astype(int), average="macro", zero_division=0), 0.0)
        print(f"    epoch {epoch}: val {'accuracy' if is_risk else 'macro-F1'} {key[0]:.4f}")
        if best_key is None or key > best_key:
            best_key, best_probs, best_epoch, bad = key, probs, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state)
    te_probs = predict_probs(model, ids_test, task, pad_id)
    del model, optimizer, scheduler, scaler, best_state
    gc.collect()
    torch.cuda.empty_cache()
    return best_probs, te_probs, dict(best_epoch=best_epoch, metric=float(best_key[0]))


def train_with_oom_retry(task, backbone, hp, ids_dev, ids_test, labels, tr, va, seed):
    """Colab GPUs differ: on out-of-memory halve the batch, double the accumulation (same effective batch), retry."""
    hp = dict(hp)
    for _ in range(4):
        try:
            return train_model(task, backbone, hp, ids_dev, ids_test, labels, tr, va, seed)
        except torch.cuda.OutOfMemoryError:
            pass
        gc.collect()
        torch.cuda.empty_cache()
        if hp["batch"] == 1:
            break
        hp["batch"], hp["accum"] = hp["batch"] // 2, hp["accum"] * 2
        print(f"    out of memory -> retrying with batch={hp['batch']}, accumulation={hp['accum']}")
    raise RuntimeError("Out of memory even at batch size 1: lower MAX_LENGTH or switch backbone.")


def fold_cache_path(task, backbone, hp, seed, k):
    cfg = dict(code=CODE_VERSION, task=task, backbone=backbone, hp=hp, seed=seed, fold=k, split=SPLIT_SIG,
               epochs=EPOCHS, patience=PATIENCE, wd=WEIGHT_DECAY, warmup=WARMUP_RATIO, llrd=LLRD, head_lr=HEAD_LR,
               dropout=DROPOUT, smoothing=LABEL_SMOOTHING, cw=RISK_CLASS_WEIGHT_POWER, pw=FACTOR_POS_WEIGHT_CAP,
               maxlen=MAX_LENGTH, head_frac=HEAD_FRAC)
    h = hashlib.md5(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:10]       # any setting change = new file
    return CACHE_DIR / f"{task}_{slug(backbone)}_s{seed}_f{k}_{h}.npz"


def run_cv(task, backbone, hp, seed, labels):
    """5-fold CV for one (backbone, seed): out-of-fold probs for the dev set + fold-averaged probs for the held-out set."""
    ids_dev, ids_test = get_encodings(backbone)
    n_out = labels.shape[1] if task == "factor" else 4
    oof = np.zeros((len(ids_dev), n_out))
    te = np.zeros((len(ids_test), n_out))
    for k, (tr, va) in enumerate(folds):
        path = fold_cache_path(task, backbone, hp, seed, k)
        if path.exists():
            z = np.load(path)
            assert np.array_equal(z["va_idx"], va), "cache file does not match the current split"
            va_p, te_p, how = z["va_probs"], z["te_probs"], f"cached (best epoch {int(z['best_epoch'])})"
        else:
            t0 = time.time()
            print(f"  {task} fold {k + 1}/{len(folds)} ({len(tr)} train / {len(va)} val)")
            va_p, te_p, info = train_with_oom_retry(task, backbone, hp, ids_dev, ids_test, labels, tr, va, seed * 100 + k)
            np.savez(path, va_idx=va, va_probs=va_p, te_probs=te_p, best_epoch=info["best_epoch"], metric=info["metric"])
            how = f"trained in {time.time() - t0:.0f}s (best epoch {info['best_epoch']})"
        oof[va] = va_p
        te += te_p / len(folds)
        print(f"  fold {k + 1}: {how}")
    return oof, te

In [ ]:
# ============================================================
# 7. RISK: 5-FOLD TRAINING (resumable) + TF-IDF MEMBER
# ============================================================
risk_members = {}        # name -> {"oof": (n_dev, 4) probs from unseen folds, "test": (n_test, 4) fold-averaged probs}

for b_i, (backbone, hp) in enumerate(BACKBONES.items()):
    try:
        get_encodings(backbone)
        for seed in SEEDS:
            name = f"{slug(backbone)}_s{seed}"
            print("\n" + "=" * 70 + f"\nRISK  |  {name}\n" + "=" * 70)
            oof, te = run_cv("risk", backbone, hp, seed, y_dev)
            risk_members[name] = {"oof": oof, "test": te}
            print(f"{name}: OOF accuracy {accuracy_score(y_dev, oof.argmax(1)):.4f}  "
                  f"macro-F1 {f1_score(y_dev, oof.argmax(1), average='macro'):.4f}")
    except Exception as e:
        if b_i == 0:
            raise                                     # the first backbone is mandatory: fix it (token / access / GPU)
        print(f"!! skipping {backbone}: {type(e).__name__}: {str(e)[:200]}")     # extra backbones are optional


def tfidf_fold(tr_texts, y_tr, other_texts):
    """Word 1-2grams + char 3-5grams -> logistic regression. Vectorisers are fitted on the fold's training posts only."""
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=150_000, sublinear_tf=True)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=200_000, sublinear_tf=True)
    X = hstack([word.fit_transform(tr_texts), char.fit_transform(tr_texts)]).tocsr()
    clf = LogisticRegression(C=3.0, max_iter=2000).fit(X, y_tr)
    outs = []
    for texts in other_texts:
        P = np.zeros((len(texts), 4))
        P[:, clf.classes_] = clf.predict_proba(hstack([word.transform(texts), char.transform(texts)]).tocsr())
        outs.append(P)
    return outs


if USE_TFIDF_MEMBER:
    print("\n" + "=" * 70 + "\nRISK  |  tfidf_lr (CPU)\n" + "=" * 70)
    oof, te = np.zeros((len(dev_df), 4)), np.zeros((len(test_df), 4))
    for k, (tr, va) in enumerate(folds):
        p_va, p_te = tfidf_fold(dev_texts[tr], y_dev[tr], [dev_texts[va], test_texts])
        oof[va] = p_va
        te += p_te / len(folds)
    risk_members["tfidf_lr"] = {"oof": oof, "test": te}
    print(f"tfidf_lr: OOF accuracy {accuracy_score(y_dev, oof.argmax(1)):.4f}  "
          f"macro-F1 {f1_score(y_dev, oof.argmax(1), average='macro'):.4f}")

assert any(n != "tfidf_lr" for n in risk_members), "No encoder was trained - read the '!! skipping' messages above."

In [ ]:
# ============================================================
# 8. RISK: ENSEMBLE + DECISION CALIBRATION  (everything fitted on OOF only)
# ============================================================
# 1) average the fine-tuned encoders; 2) blend in TF-IDF with a weight chosen by OOF log-loss;
# 3) learn 4 additive class biases that maximise OOF ACCURACY (undoes the shift caused by class weighting).
# "Cross-fitted" = for each fold, weight and biases are fitted on the OTHER folds, so the reported OOF accuracy is honest.
CLASSES = [0, 1, 2, 3]


def _logp(p):
    return np.log(np.clip(p, 1e-9, 1.0))


def fit_bias(logp, y, grid=np.round(np.arange(-1.0, 1.0001, 0.05), 2), passes=3):
    b = np.zeros(logp.shape[1])
    for _ in range(passes):
        changed = False
        for c in range(logp.shape[1]):
            scores = []
            for g in grid:
                b2 = b.copy()
                b2[c] = g
                scores.append(((logp + b2).argmax(1) == y).mean() - 1e-6 * abs(g))     # tiny penalty: ties -> smaller shift
            best = grid[int(np.argmax(scores))]
            if best != b[c]:
                b[c], changed = best, True
        if not changed:
            break
    return b


def pick_blend_weight(P_main, P_tf, y, grid=np.round(np.arange(0.0, 0.5001, 0.05), 2)):
    losses = [log_loss(y, (1 - w) * P_main + w * P_tf, labels=CLASSES) for w in grid]
    return float(grid[int(np.argmin(losses))])


def blend(P_main, P_tf, w):
    return P_main if P_tf is None else (1 - w) * P_main + w * P_tf


def crossfit(P_main, P_tf, y, fold_ids):
    pred_plain, pred_bias, weights = np.zeros(len(y), int), np.zeros(len(y), int), []
    for k in np.unique(fold_ids):
        fit, hold = fold_ids != k, fold_ids == k
        w = pick_blend_weight(P_main[fit], P_tf[fit], y[fit]) if P_tf is not None else 0.0
        lp = _logp(blend(P_main, P_tf, w))
        b = fit_bias(lp[fit], y[fit])
        pred_plain[hold] = lp[hold].argmax(1)
        pred_bias[hold] = (lp[hold] + b).argmax(1)
        weights.append(w)
    return pred_plain, pred_bias, weights


main_names = [n for n in risk_members if n != "tfidf_lr"]          # the fine-tuned encoders
P_main_oof = np.mean([risk_members[n]["oof"] for n in main_names], axis=0)
P_main_test = np.mean([risk_members[n]["test"] for n in main_names], axis=0)
has_tf = "tfidf_lr" in risk_members
P_tf_oof = risk_members["tfidf_lr"]["oof"] if has_tf else None
P_tf_test = risk_members["tfidf_lr"]["test"] if has_tf else None

pred_plain_cf, pred_bias_cf, cf_weights = crossfit(P_main_oof, P_tf_oof, y_dev, fold_id)
acc_plain, acc_bias = accuracy_score(y_dev, pred_plain_cf), accuracy_score(y_dev, pred_bias_cf)
use_bias = acc_bias > acc_plain                    # keep the biases only if they help on unseen folds
risk_oof_pred = pred_bias_cf if use_bias else pred_plain_cf

blend_w = pick_blend_weight(P_main_oof, P_tf_oof, y_dev) if has_tf else 0.0
blend_oof = blend(P_main_oof, P_tf_oof, blend_w)
blend_test = blend(P_main_test, P_tf_test, blend_w)
risk_bias = fit_bias(_logp(blend_oof), y_dev) if use_bias else np.zeros(4)
risk_test_pred = (_logp(blend_test) + risk_bias).argmax(1)       # decisions frozen here; the held-out labels were not touched
risk_test_labels = [ID_TO_RISK[i] for i in risk_test_pred]
_z = _logp(blend_test) + risk_bias
risk_test_confidence = np.exp(_z - _z.max(1, keepdims=True))
risk_test_confidence = (risk_test_confidence / risk_test_confidence.sum(1, keepdims=True)).max(1)

ablation = []
for n, m in risk_members.items():
    p = m["oof"].argmax(1)
    ablation.append(dict(stage=n, oof_acc=accuracy_score(y_dev, p), oof_macro_f1=f1_score(y_dev, p, average="macro"),
                         test_pred=m["test"].argmax(1)))
if len(main_names) > 1:
    p = P_main_oof.argmax(1)
    ablation.append(dict(stage=f"encoder average ({len(main_names)} models)", oof_acc=accuracy_score(y_dev, p),
                         oof_macro_f1=f1_score(y_dev, p, average="macro"), test_pred=P_main_test.argmax(1)))
if has_tf:
    ablation.append(dict(stage=f"+ tfidf blend (w={blend_w:.2f}, cross-fitted)", oof_acc=acc_plain,
                         oof_macro_f1=f1_score(y_dev, pred_plain_cf, average="macro"), test_pred=blend_test.argmax(1)))
ablation.append(dict(stage="+ class biases (cross-fitted)" if use_bias else "final (class biases did not help -> off)",
                     oof_acc=accuracy_score(y_dev, risk_oof_pred), oof_macro_f1=f1_score(y_dev, risk_oof_pred, average="macro"),
                     test_pred=risk_test_pred))

print("Risk OOF accuracy by stage (OOF = dev posts scored by models that never saw their user):")
for a in ablation:
    print(f"  {a['stage']:<45} accuracy {a['oof_acc']:.4f} | macro-F1 {a['oof_macro_f1']:.4f}")
print(f"\nblend weight (tfidf): {blend_w:.2f} | class biases {np.round(risk_bias, 2).tolist()} | biases used: {use_bias}")

In [ ]:
# ============================================================
# 9. FACTORS (24-label): 5-FOLD, PER-FACTOR THRESHOLDS TUNED ON OOF
# ============================================================
factor_oof_true = Y_factor_dev.astype(int)
factor_test_true = Y_factor_test.astype(int)
factor_oof_pred = factor_test_pred = factor_thresholds = None

THRESH_GRID = np.round(np.arange(0.10, 0.9001, 0.02), 2)
MIN_POS = 8                  # a factor with fewer positives keeps 0.5: tuning on a handful of cases only overfits


def tune_thresholds(P, Y):
    thr = np.full(Y.shape[1], 0.5)
    for j in range(Y.shape[1]):
        if Y[:, j].sum() < MIN_POS:
            continue
        scores = [f1_score(Y[:, j], (P[:, j] >= t).astype(int), zero_division=0) for t in THRESH_GRID]
        thr[j] = THRESH_GRID[int(np.argmax(scores))]
    return thr


if RUN_FACTORS:
    hp_f = BACKBONES.get(FACTOR_BACKBONE, dict(lr=3e-5, batch=8, accum=2))
    print("=" * 70 + f"\nFACTORS  |  {slug(FACTOR_BACKBONE)}\n" + "=" * 70)
    factor_oof_probs, factor_test_probs = run_cv("factor", FACTOR_BACKBONE, hp_f, SEEDS[0], Y_factor_dev)

    # honest OOF predictions: each fold is thresholded with values tuned on the OTHER folds
    factor_oof_pred = np.zeros_like(factor_oof_true)
    for k in range(N_FOLDS):
        fit, hold = fold_id != k, fold_id == k
        thr_k = tune_thresholds(factor_oof_probs[fit], factor_oof_true[fit])
        factor_oof_pred[hold] = (factor_oof_probs[hold] >= thr_k).astype(int)
    factor_thresholds = tune_thresholds(factor_oof_probs, factor_oof_true)
    factor_test_pred = (factor_test_probs >= factor_thresholds).astype(int)

    for avg in ("micro", "macro", "samples"):
        print(f"Factor {avg:<7} F1 (OOF, cross-fitted thresholds): "
              f"{f1_score(factor_oof_true, factor_oof_pred, average=avg, zero_division=0):.4f}")
else:
    print("RUN_FACTORS = False -> factor step skipped.")

In [ ]:
# ============================================================
# 10. EVIDENCE (Groq gpt-oss-120b): verbatim spans only
# ============================================================
n_test = len(test_df)
evidence_predictions = [""] * n_test
evidence_sources = ["none"] * n_test
evidence_coverage = evidence_llm_coverage = evidence_token_f1 = None

# Edit these to match the competition's own definitions.
RISK_DEFINITIONS = {
    "indicator": "signs of distress or risk factors (hopelessness, depression, isolation) WITHOUT explicit thoughts of suicide",
    "ideation":  "thoughts of wanting to die or to kill oneself, without a plan or any action",
    "behavior":  "self-harm, or planning/preparing a suicide (means, method, note, rehearsal) without an actual attempt",
    "attempt":   "describes an actual suicide attempt (recent or past)",
}
CUES = {   # only used by the sentence fallback
    "indicator": r"hopeless|worthless|depress|lonely|empty|exhaust|tired of|burden|no point|nothing matters|can.?t go on|give up",
    "ideation":  r"suicid|kill myself|want to die|wanna die|end it all|end my life|better off dead|don.?t want to (?:live|be here|exist)|wish i (?:was|were) dead",
    "behavior":  r"plan|pills|rope|gun|razor|knife|bridge|note|goodbye|method|cutting|self.?harm|hurt myself|prepar|bought|stockpil",
    "attempt":   r"attempt|overdos|tried to (?:kill|end|take|hang)|swallowed|survived|hospital|stomach pump|woke up in|cut my wrists?|slit",
}

# ---- matching that survives curly quotes / line breaks and returns the ORIGINAL characters ----
_FOLD = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-", "\u2026": "..."}


def _fold(s):
    chars, idx, prev_space = [], [], False
    for i, ch in enumerate(s):
        ch = _FOLD.get(ch, ch)
        if ch.isspace():
            if prev_space:
                continue
            ch, prev_space = " ", True
        else:
            prev_space = False
        for c in ch.lower():
            chars.append(c)
            idx.append(i)
    return "".join(chars), idx


def find_verbatim(span, text, text_f, idx):
    span_f = _fold(span.strip().strip("\"'` "))[0].strip()
    if len(span_f) < 3:
        return None
    pos = text_f.find(span_f)
    if pos < 0:
        return None
    return text[idx[pos]: idx[pos + len(span_f) - 1] + 1]


def parse_spans(raw):
    if not raw:
        return []
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip())
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if m:
        try:
            v = json.loads(m.group(0)).get("evidence", [])
            return [str(x) for x in ([v] if isinstance(v, str) else v)]
        except Exception:
            pass
    parts = re.split(r";|\n", raw)                                  # model ignored the JSON format
    return [re.sub(r"^\s*(?:[-*\u2022]|\d+[.)])\s*", "", p) for p in parts if p.strip()]


def build_messages(text, risk):
    system = "You label evidence spans for a research dataset about mental-health text. Reply with JSON only."
    user = f"""Predicted risk level: {risk} ({RISK_DEFINITIONS[risk]})

Post:
<<<
{text}
>>>

Pick 1 to 3 SHORT spans (each at most 25 words) from the post that best support this risk level.
Rules:
- Copy each span EXACTLY, character for character, from the post. No paraphrasing, no ellipses.
- Prefer the most direct statements about the risk level over background detail.
- Output ONLY this JSON: {{"evidence": ["span 1", "span 2"]}}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def call_groq(messages):
    """gpt-oss is a reasoning model: reasoning tokens count against the limit (old max_tokens=200 left no room for the answer)."""
    use_effort, last = True, None
    for attempt in range(5):
        try:
            kw = dict(model=GROQ_MODEL, messages=messages, temperature=0, max_completion_tokens=1500)
            if use_effort:
                kw["extra_body"] = {"reasoning_effort": "low"}
            r = groq_client.chat.completions.create(**kw)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            last, code = e, getattr(e, "status_code", None)
            if code == 400 and use_effort:                          # this API/model rejected reasoning_effort: retry without it
                use_effort = False
                continue
            if code in (400, 401, 403, 404) or isinstance(e, TypeError):
                break                                               # retrying cannot help
            wait = 2 ** attempt
            if code == 429:
                try:
                    wait = max(wait, float(e.response.headers.get("retry-after", 0)))
                except Exception:
                    pass
            time.sleep(min(wait, 60))
    print("  Groq call failed:", str(last)[:120])
    return ""


def fallback_sentence(text, risk):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text) if len(s.split()) >= 3]
    if not sents:
        return text[:200].strip()
    pat = re.compile(CUES[risk], re.I)
    return max(sents, key=lambda s: (len(pat.findall(s)), -abs(len(s.split()) - 18)))[:300].strip()


def extract_evidence(text, risk):
    """Returns (evidence string, source) with source in {llm, fallback, none}. Every returned span is a substring of the post."""
    shown = text if len(text) <= MAX_POST_CHARS else text[:MAX_POST_CHARS // 2] + "\n[...]\n" + text[-(MAX_POST_CHARS // 2):]
    text_f, idx = _fold(text)
    found = []
    for span in parse_spans(call_groq(build_messages(shown, risk))):
        hit = find_verbatim(span, text, text_f, idx)
        if hit and hit not in found:
            found.append(hit)
    if found:
        return "; ".join(found[:3]), "llm"
    if EVIDENCE_FALLBACK:
        return fallback_sentence(text, risk), "fallback"
    return "", "none"


def tok_f1(pred, gold_list):
    p = Counter(re.findall(r"\w+", pred.lower()))
    g = Counter(re.findall(r"\w+", " ".join(gold_list).lower()))
    overlap = sum((p & g).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / sum(p.values()), overlap / sum(g.values())
    return 2 * precision * recall / (precision + recall)


if RUN_EVIDENCE:
    cache_path = CACHE_DIR / "evidence_cache.json"                  # only real LLM answers are cached: re-runs cost no API calls
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    lock, new_calls = threading.Lock(), [0]
    n_do = n_test if EVIDENCE_N is None else min(EVIDENCE_N, n_test)

    def work(i):
        text, risk = test_df.at[i, "text_original"], risk_test_labels[i]
        k = hashlib.md5((risk + "\x00" + text).encode()).hexdigest()
        if k in cache:
            return i, tuple(cache[k])
        res = extract_evidence(text, risk)
        if res[1] == "llm":
            with lock:
                cache[k] = list(res)
                new_calls[0] += 1
                if new_calls[0] % 25 == 0:
                    cache_path.write_text(json.dumps(cache))
        return i, res

    with ThreadPoolExecutor(EVIDENCE_WORKERS) as ex:
        for i, (ev, src) in tqdm(ex.map(work, range(n_do)), total=n_do, desc="evidence"):
            evidence_predictions[i], evidence_sources[i] = ev, src
    cache_path.write_text(json.dumps(cache))

    done = range(n_do)
    evidence_coverage = float(np.mean([bool(evidence_predictions[i].strip()) for i in done]))
    evidence_llm_coverage = float(np.mean([evidence_sources[i] == "llm" for i in done]))
    with_gold = [i for i in done if test_df.at[i, "evidence_list"]]
    if with_gold:
        evidence_token_f1 = float(np.mean([tok_f1(evidence_predictions[i], test_df.at[i, "evidence_list"]) for i in with_gold]))
    print(f"\nEvidence coverage (non-empty): {evidence_coverage:.2%} | of which straight from the LLM: {evidence_llm_coverage:.2%}"
          + (f" | token-F1 vs gold evidence: {evidence_token_f1:.3f}" if evidence_token_f1 is not None else ""))
else:
    print("Evidence step skipped.")

In [ ]:
# ============================================================
# 11. HELD-OUT SCORING (once, after every choice above was frozen) + OUTPUT FILES
# ============================================================
import shutil

oof_acc = accuracy_score(y_dev, risk_oof_pred)
oof_macro = f1_score(y_dev, risk_oof_pred, average="macro", zero_division=0)
oof_weighted = f1_score(y_dev, risk_oof_pred, average="weighted", zero_division=0)
test_acc = accuracy_score(y_test, risk_test_pred)
test_macro = f1_score(y_test, risk_test_pred, average="macro", zero_division=0)
test_weighted = f1_score(y_test, risk_test_pred, average="weighted", zero_division=0)

print("=" * 70)
print("RISK ACCURACY")
print("=" * 70)
print(f"OOF, cross-fitted : {oof_acc:.2%}   (macro-F1 {oof_macro:.2%}, weighted-F1 {oof_weighted:.2%})")
print(f"HELD-OUT 20%      : {test_acc:.2%}   (macro-F1 {test_macro:.2%}, weighted-F1 {test_weighted:.2%})   target {TARGET_ACC:.0%}")
print("OOF is slightly optimistic (each fold's best epoch was picked on that same fold). Quote the HELD-OUT number.")
print("\nStage by stage (held-out numbers are for reading only: no choice above used them):")
for a in ablation:
    print(f"  {a['stage']:<45} OOF acc {a['oof_acc']:.4f} | held-out acc {accuracy_score(y_test, a['test_pred']):.4f}")

factor_metrics_test = {}
if factor_test_pred is not None:
    for avg in ("micro", "macro", "samples"):
        factor_metrics_test[avg] = f1_score(factor_test_true, factor_test_pred, average=avg, zero_division=0)
    print("\nFactor F1 on held-out: " + " | ".join(f"{k} {v:.4f}" for k, v in factor_metrics_test.items()))

# ---------------- files ----------------
factor_text = (["; ".join(ID_TO_FACTOR[j] for j in np.where(row == 1)[0]) for row in factor_test_pred]
               if factor_test_pred is not None else [""] * len(test_df))
results = pd.DataFrame({
    "post": test_df["text_original"],
    "predicted_suicide_risk": risk_test_labels,
    "predicted_factors": factor_text,
    "predicted_evidence": evidence_predictions,
})
if USER_COL is not None:
    results[USER_COL] = test_df[USER_COL].values
results.to_csv("heldout_predictions.csv", index=False)

analysis = pd.DataFrame({
    "post": test_df["text_original"],
    "true_risk": test_df["risk_label"],
    "predicted_risk": risk_test_labels,
    "correct": (y_test == risk_test_pred),
    "confidence": risk_test_confidence,
    **{f"p_{r}": blend_test[:, i] for i, r in enumerate(RISK_ORDER)},
    "evidence_source": evidence_sources,
})
analysis.to_csv("heldout_analysis.csv", index=False)          # sort by confidence among wrong rows for error analysis

if factor_thresholds is not None:
    pd.DataFrame({"factor": factor_names, "threshold": factor_thresholds}).to_csv("factor_thresholds.csv", index=False)

metrics = {
    "risk_oof_accuracy": oof_acc, "risk_oof_macro_f1": oof_macro, "risk_oof_weighted_f1": oof_weighted,
    "risk_heldout_accuracy": test_acc, "risk_heldout_macro_f1": test_macro, "risk_heldout_weighted_f1": test_weighted,
    "factor_oof_micro_f1": f1_score(factor_oof_true, factor_oof_pred, average="micro", zero_division=0) if factor_oof_pred is not None else None,
    "factor_oof_macro_f1": f1_score(factor_oof_true, factor_oof_pred, average="macro", zero_division=0) if factor_oof_pred is not None else None,
    "factor_oof_samples_f1": f1_score(factor_oof_true, factor_oof_pred, average="samples", zero_division=0) if factor_oof_pred is not None else None,
    "factor_heldout_micro_f1": factor_metrics_test.get("micro"),
    "factor_heldout_macro_f1": factor_metrics_test.get("macro"),
    "factor_heldout_samples_f1": factor_metrics_test.get("samples"),
    "evidence_coverage": evidence_coverage, "evidence_llm_coverage": evidence_llm_coverage, "evidence_token_f1": evidence_token_f1,
}
pd.DataFrame([{"metric": k, "value": v} for k, v in metrics.items() if v is not None]).to_csv("model_metrics.csv", index=False)

out_files = ["heldout_predictions.csv", "heldout_analysis.csv", "model_metrics.csv"] + (["factor_thresholds.csv"] if factor_thresholds is not None else [])
if str(CACHE_DIR).startswith("/content/drive"):
    for f in out_files:
        shutil.copy(f, CACHE_DIR / f)                         # a copy on Drive survives a runtime reset
print("\nSaved:", ", ".join(out_files))

# ---------------- variables read by the evaluation cell below ----------------
risk_oof_true, risk_oof_groups, risk_oof_folds = y_dev, groups_dev, fold_id
risk_test_true, risk_test_groups = y_test, groups_test

print("\nSample held-out predictions:")
for i in range(min(5, len(results))):
    print("-" * 70)
    print("POST    :", results.iloc[i]["post"][:400].replace("\n", " "))
    print("TRUE    :", test_df.iloc[i]["risk_label"], "| PRED:", results.iloc[i]["predicted_suicide_risk"], f"({risk_test_confidence[i]:.2f})")
    print("FACTORS :", results.iloc[i]["predicted_factors"])
    print("EVIDENCE:", results.iloc[i]["predicted_evidence"][:300])

In [ ]:
# ============================================================
# FINAL EVALUATION  --  risk (4-class), factors (24-label), evidence
# ------------------------------------------------------------
# Every number printed here is computed from predictions that already
# exist in the notebook. Nothing is hard-coded: if the required arrays
# are missing, the cell stops with an error instead of printing stale
# metrics. Put it LAST, after the training / CV / evidence cells.
# ============================================================
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    cohen_kappa_score, confusion_matrix, f1_score,
    precision_recall_fscore_support,
)

try:
    from IPython.display import display
except ImportError:                          # plain-python fallback
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

# ---------------- config: edit the names here if yours differ ----------------
TARGET_ACC    = 0.80
RISK_NAMES    = ["Indicator", "Ideation", "Behavior", "Attempt"]     # ids 0..3
N_BOOT, SEED  = 2000, 42
SHOW_PIPELINE = True
RESULTS_PATH  = "final_metrics.json"

# (title, [(y_true_var, y_pred_var), ...], [groups_var, ...], [folds_var, ...])
RISK_SPLITS = [
    ("OOF (cross-fitted, group CV)",
     [("risk_oof_true", "risk_oof_pred"), ("risk_oof_labels", "risk_oof_preds")],
     ["risk_oof_groups"], ["risk_oof_folds"]),
    ("Held-out test (20%)",
     [("risk_test_true", "risk_test_pred"), ("risk_test_labels", "risk_test_preds")],
     ["risk_test_groups"], []),
]
FACTOR_PAIRS     = [("factor_oof_true", "factor_oof_pred"),
                    ("factor_oof_labels", "factor_oof_preds")]      # (n, 24) binary 0/1
FACTOR_NAME_VARS = ["factor_names", "factor_cols", "FACTOR_NAMES"]
EVIDENCE_VAR     = "evidence_coverage"                              # fraction in [0, 1]

PIPELINE = """
                         DATASET
                            │
                            ▼
                   ┌─────────────────┐
                   │   DATA LOADING  │
                   │    train.xlsx   │
                   └────────┬────────┘
                            │
                            ▼
                   ┌─────────────────┐
                   │       EDA       │
                   │ Risk / Length / │
                   │     Factors     │
                   └────────┬────────┘
                            │
                            ▼
                   ┌─────────────────┐
                   │ PREPROCESSING   │
                   │ • Text Cleaning │
                   │ • Factor Parsing│
                   │ • Evidence Parse│
                   └────────┬────────┘
                            │
                            ▼
              ┌─────────────────────────────┐
              │ Group-Aware 80/20 Split     │
              │ + 5-Fold Cross Validation   │
              └──────────────┬──────────────┘
                             │
                 ┌───────────┴───────────┐
                 ▼                       ▼
       ┌───────────────────┐   ┌───────────────────┐
       │ Encoder Ensemble  │   │  Mental-RoBERTa   │
       │ + OOF Calibration │   │ Factor Classifier │
       └─────────┬─────────┘   └─────────┬─────────┘
                 │                       │
                 ▼                       ▼
          4 Risk Classes            24 Factors
          • Indicator               Multi-label
          • Ideation
          • Behavior
          • Attempt
                 │                       │
                 ▼                       ▼
           Risk Metrics             Factor Metrics
           Accuracy                  Micro F1
           Macro F1                  Macro F1
           Weighted F1               Samples F1
                 │                       │
                 └───────────┬───────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │  GPT-OSS-120B   │
                    │ Evidence Extract│
                    └────────┬────────┘
                             │
                             ▼
                     Verbatim Evidence
                             │
                             ▼
                       FINAL OUTPUT
"""


# ============================================================
# helpers
# ============================================================
def _lookup(names, n=None):
    """First notebook variable in `names` that exists, is not None, and (if n is given) has length n."""
    for name in names:
        val = globals().get(name)
        if val is None:
            continue
        if n is None or len(val) == n:
            return val
        print(f"note: {name} has length {len(val)}, expected {n} -> ignored")
    return None


def _find_pair(pairs):
    for t, p in pairs:
        if globals().get(t) is not None and globals().get(p) is not None:     # a skipped step leaves None behind
            return globals()[t], globals()[p]
    return None


def _to_class_ids(values, what):
    a = np.asarray(values)
    if a.ndim != 1:
        raise ValueError(f"{what}: expected 1-D class ids, got shape {a.shape}. "
                         "Pass argmax(logits), not the logits themselves.")
    if a.dtype.kind in "OUS":                                   # strings -> ids
        ids = {n.lower(): i for i, n in enumerate(RISK_NAMES)}
        try:
            return np.array([ids[str(v).strip().lower()] for v in a])
        except KeyError as e:
            raise ValueError(f"{what}: unknown class {e}; expected {RISK_NAMES}") from None
    if a.dtype.kind == "f" and not np.array_equal(a, np.round(a)):
        raise ValueError(f"{what}: non-integer floats look like probabilities; pass argmax.")
    a = a.astype(int)
    if a.min() < 0 or a.max() >= len(RISK_NAMES):
        raise ValueError(f"{what}: ids must be in 0..{len(RISK_NAMES) - 1}, got {a.min()}..{a.max()}")
    return a


def _bootstrap_ci(correct, groups=None, n_boot=N_BOOT, seed=SEED):
    """95% bootstrap CI for accuracy. With `groups`, whole users are resampled."""
    rng = np.random.default_rng(seed)
    correct = np.asarray(correct, dtype=float)
    if groups is None:
        hits, size = correct, np.ones_like(correct)
    else:
        _, inv = np.unique(np.asarray(groups), return_inverse=True)
        hits = np.bincount(inv, weights=correct)
        size = np.bincount(inv).astype(float)
    g, accs = len(hits), np.empty(n_boot)
    for s in range(0, n_boot, 100):
        e = min(s + 100, n_boot)
        idx = rng.integers(0, g, size=(e - s, g))
        accs[s:e] = hits[idx].sum(axis=1) / size[idx].sum(axis=1)
    return tuple(float(x) for x in np.percentile(accs, [2.5, 97.5]))


def _json_default(o):
    if isinstance(o, pd.DataFrame):
        return o.round(6).to_dict(orient="index")
    if hasattr(o, "tolist"):
        return o.tolist()
    return str(o)


# ============================================================
# metrics (pure computation; validates inputs before anything is shown)
# ============================================================
def risk_metrics(title, y_true, y_pred, groups=None, folds=None):
    yt = _to_class_ids(y_true, f"{title}: y_true")
    yp = _to_class_ids(y_pred, f"{title}: y_pred")
    if len(yt) != len(yp):
        raise ValueError(f"{title}: {len(yt)} true labels vs {len(yp)} predictions")
    ids = list(range(len(RISK_NAMES)))

    m = {"title": title, "n": int(len(yt)), "grouped_ci": groups is not None}
    m["accuracy"] = float(accuracy_score(yt, yp))
    m["acc_ci95"] = _bootstrap_ci(yt == yp, groups)
    m["balanced_accuracy"] = float(balanced_accuracy_score(yt, yp))
    m["macro_f1"] = float(f1_score(yt, yp, labels=ids, average="macro", zero_division=0))
    m["weighted_f1"] = float(f1_score(yt, yp, labels=ids, average="weighted", zero_division=0))
    m["quadratic_kappa"] = float(cohen_kappa_score(yt, yp, weights="quadratic"))

    p, r, f, s = precision_recall_fscore_support(yt, yp, labels=ids, zero_division=0)
    m["per_class"] = pd.DataFrame({"Precision": p, "Recall": r, "F1": f, "Support": s},
                                  index=RISK_NAMES)
    cm = confusion_matrix(yt, yp, labels=ids)
    m["cm"] = cm
    m["cm_norm"] = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    wrong = yt != yp
    m["adjacent_error_share"] = (float((np.abs(yt - yp)[wrong] == 1).mean())
                                 if wrong.any() else float("nan"))
    off_diag = [(int(cm[i, j]), i, j) for i in ids for j in ids if i != j]
    m["top_confusions"] = sorted(off_diag, reverse=True)[:4]

    if folds is not None:
        folds = np.asarray(folds)
        fl = np.unique(folds)
        m["folds"] = {
            "ids": fl.tolist(),
            "accuracy": [float(accuracy_score(yt[folds == k], yp[folds == k])) for k in fl],
            "macro_f1": [float(f1_score(yt[folds == k], yp[folds == k], labels=ids,
                                        average="macro", zero_division=0)) for k in fl],
        }
    return m


def factor_metrics(y_true, y_pred):
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    if yt.ndim != 2 or yt.shape != yp.shape:
        raise ValueError(f"factors: need matching (n, n_factors) arrays, got {yt.shape} and {yp.shape}")
    for nm, a in (("y_true", yt), ("y_pred", yp)):
        if not np.isin(a, (0, 1)).all():
            raise ValueError(f"factors {nm} must be binary 0/1 - apply your threshold before this cell")
    names = _lookup(FACTOR_NAME_VARS, yt.shape[1])
    names = list(names) if names is not None else [f"factor_{i}" for i in range(yt.shape[1])]
    yt, yp = yt.astype(int), yp.astype(int)
    scores = {f"{k}_f1": float(f1_score(yt, yp, average=k, zero_division=0))
              for k in ("micro", "macro", "samples")}
    per = pd.DataFrame({"F1": f1_score(yt, yp, average=None, zero_division=0),
                        "Support": yt.sum(axis=0), "Predicted": yp.sum(axis=0)},
                       index=names).sort_values("F1")
    return scores, per


# ============================================================
# reporting (prints + plots)
# ============================================================
def show_risk(m):
    lo, hi = m["acc_ci95"]
    if m["accuracy"] >= TARGET_ACC and lo >= TARGET_ACC:
        verdict = "target reached (the whole 95% CI is above it)"
    elif m["accuracy"] >= TARGET_ACC:
        verdict = "point estimate at/above the target, but the 95% CI reaches below it"
    else:
        verdict = f"{(TARGET_ACC - m['accuracy']) * 100:.2f} pts below the {TARGET_ACC:.0%} target"
    ci_note = ("users resampled" if m["grouped_ci"]
               else "posts treated as independent: optimistic if users have several posts")
    print("=" * 70)
    print(f"RISK CLASSIFICATION | {m['title']} | n = {m['n']}")
    print("=" * 70)
    print(f"Accuracy          : {m['accuracy']:.2%}   95% CI {lo:.2%} to {hi:.2%} ({ci_note})")
    print(f"                    -> {verdict}")
    print(f"Balanced accuracy : {m['balanced_accuracy']:.2%}")
    print(f"Macro F1          : {m['macro_f1']:.2%}")
    print(f"Weighted F1       : {m['weighted_f1']:.2%}")
    print(f"Quadratic kappa   : {m['quadratic_kappa']:.3f}  (classes treated as ordered)")
    if "folds" in m:
        fa, ff = np.array(m["folds"]["accuracy"]), np.array(m["folds"]["macro_f1"])
        print(f"Per-fold accuracy : {fa.mean():.2%} +/- {fa.std(ddof=1):.2%}   "
              f"macro F1: {ff.mean():.2%} +/- {ff.std(ddof=1):.2%}   ({len(fa)} folds)")
    print("\nPer-class metrics:")
    display(m["per_class"].round(4))
    print("Biggest confusions (true -> predicted):")
    for cnt, i, j in m["top_confusions"]:
        print(f"  {RISK_NAMES[i]:>9} -> {RISK_NAMES[j]:<9} {cnt:5d}  "
              f"({m['cm_norm'][i, j]:.1%} of {RISK_NAMES[i]})")
    print(f"Errors that are one step away on the ordered scale: {m['adjacent_error_share']:.1%}\n")

    fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))
    for ax, mat, fmt, ttl in ((axes[0], m["cm"], "d", "Counts"),
                              (axes[1], m["cm_norm"], ".0%", "Row-normalised (recall)")):
        ConfusionMatrixDisplay(mat, display_labels=RISK_NAMES).plot(
            ax=ax, values_format=fmt, colorbar=False, cmap="Blues")
        ax.set_title(f"Confusion matrix - {ttl}")
        ax.tick_params(axis="x", rotation=20)
    x, w = np.arange(len(RISK_NAMES)), 0.26
    for k, col in enumerate(("Precision", "Recall", "F1")):
        axes[2].bar(x + (k - 1) * w, m["per_class"][col] * 100, w, label=col)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels([f"{n}\n(n={int(s)})" for n, s in
                             zip(RISK_NAMES, m["per_class"]["Support"])])
    axes[2].set_ylim(0, 100)
    axes[2].set_ylabel("Score (%)")
    axes[2].set_title("Per-class precision / recall / F1")
    axes[2].legend()
    fig.suptitle(f"Risk classification - {m['title']}")
    plt.tight_layout()
    plt.show()


def show_factors(scores, per):
    print("=" * 70)
    print("FACTOR IDENTIFICATION (24-label, OOF)")
    print("=" * 70)
    for k in ("micro", "macro", "samples"):
        print(f"{k.capitalize():<8} F1 : {scores[f'{k}_f1']:.2%}")
    print("\nWeakest factors:")
    display(per.head(8).round(4))

    fig, ax = plt.subplots(figsize=(9, 0.32 * len(per) + 1.5))
    ax.barh(per.index, per["F1"] * 100)
    for y, (f1v, sup) in enumerate(zip(per["F1"], per["Support"])):
        ax.text(f1v * 100 + 1, y, f"{f1v:.0%}  (n={int(sup)})", va="center", fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_xlabel("F1 (%)")
    ax.set_title("Per-factor F1 (weakest at top)")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


# ============================================================
# run: compute everything first, then report
# ============================================================
final_results = {"generated": datetime.now().isoformat(timespec="seconds"),
                 "target_accuracy": TARGET_ACC, "risk": {}, "factors": None,
                 "evidence_coverage": None}

risk_reports = []
for title, pairs, group_vars, fold_vars in RISK_SPLITS:
    hit = _find_pair(pairs)
    if hit is not None:
        n = len(hit[0])
        risk_reports.append(risk_metrics(title, *hit,
                                         groups=_lookup(group_vars, n),
                                         folds=_lookup(fold_vars, n)))
if not risk_reports:
    wanted = "; ".join(f"{t} + {p}" for _, pairs, _, _ in RISK_SPLITS for t, p in pairs)
    raise RuntimeError("No risk predictions found in the notebook. Define one of: " + wanted)

factor_pair = _find_pair(FACTOR_PAIRS)
factor_scores, per_factor = factor_metrics(*factor_pair) if factor_pair else (None, None)

evidence_cov = globals().get(EVIDENCE_VAR)
if evidence_cov is not None:
    evidence_cov = float(evidence_cov)
    if not 0.0 <= evidence_cov <= 1.0:
        raise ValueError(f"{EVIDENCE_VAR} must be a fraction in [0, 1], got {evidence_cov}")

if SHOW_PIPELINE:
    print("=" * 70)
    print("FINAL MODEL PIPELINE")
    print("=" * 70)
    print(PIPELINE)

for m in risk_reports:
    show_risk(m)
    final_results["risk"][m["title"]] = m
if len(risk_reports) == 1 and risk_reports[0]["title"].startswith("OOF"):
    print("note: no held-out test predictions found - only OOF is reported. Score the test "
          "split once, after every modelling choice is frozen.\n")
if factor_scores:
    show_factors(factor_scores, per_factor)
    final_results["factors"] = {**factor_scores, "per_factor": per_factor}
else:
    print("Factor metrics: skipped (no factor predictions found).\n")
if evidence_cov is not None:
    final_results["evidence_coverage"] = evidence_cov
else:
    print(f"Evidence coverage: skipped ({EVIDENCE_VAR} not defined by the evidence cell).\n")

# ---------------- summary table ----------------
rows = []
for m in risk_reports:
    for label, key in (("Accuracy", "accuracy"), ("Balanced accuracy", "balanced_accuracy"),
                       ("Macro F1", "macro_f1"), ("Weighted F1", "weighted_f1")):
        rows.append(("Risk classification", m["title"], label, m[key]))
if factor_scores:
    for label, key in (("Micro F1", "micro_f1"), ("Macro F1", "macro_f1"), ("Samples F1", "samples_f1")):
        rows.append(("Factor identification", "OOF", label, factor_scores[key]))
if evidence_cov is not None:
    rows.append(("Evidence extraction", "-", "Evidence coverage", evidence_cov))
metrics_df = pd.DataFrame(rows, columns=["Task", "Split", "Metric", "Score"])
metrics_df["Score (%)"] = (metrics_df["Score"] * 100).round(2)
print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
display(metrics_df)

# ---------------- overview chart (first split + factors + evidence) ----------------
first = risk_reports[0]
bars = [("Risk\nAccuracy", first["accuracy"]), ("Risk\nMacro F1", first["macro_f1"]),
        ("Risk\nWeighted F1", first["weighted_f1"])]
if factor_scores:
    bars += [("Factor\nMicro F1", factor_scores["micro_f1"]),
             ("Factor\nMacro F1", factor_scores["macro_f1"]),
             ("Factor\nSamples F1", factor_scores["samples_f1"])]
if evidence_cov is not None:
    bars.append(("Evidence\nCoverage", evidence_cov))
fig, ax = plt.subplots(figsize=(11, 5.5))
rects = ax.bar([b[0] for b in bars], [b[1] * 100 for b in bars])
ax.axhline(TARGET_ACC * 100, ls="--", lw=1, color="crimson",
           label=f"Risk accuracy target ({TARGET_ACC:.0%})")
for rect, (_, v) in zip(rects, bars):
    ax.text(rect.get_x() + rect.get_width() / 2, v * 100 + 1, f"{v:.2%}", ha="center")
ax.set_ylim(0, 100)
ax.set_ylabel("Score (%)")
ax.set_title(f"Overall model performance ({first['title']})")
ax.legend()
plt.tight_layout()
plt.show()

with open(RESULTS_PATH, "w") as fh:
    json.dump(final_results, fh, indent=2, default=_json_default)
print(f"All numbers above were computed from the predictions in memory; saved to {RESULTS_PATH}")